# Stage 2: Text-Conditioned Motion Continuation

Train a GRU to predict the next absolute feature frame with text conditioning.

- Input: motion sequence (normalized) + CLIP text embedding (512)
- Output: predicted next frame
- Loss: MSE (masked for padding)
- Autoregressive inference from the first frame with text input

In [10]:
# Imports
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from pathlib import Path

from config import Config
from utils.dataset import Text2MotionDataset, text2motion_collate_fn
from utils.motion_utils import t2m_kinematic_chain
from utils.text_encoder import CLIPEncoder
from utils.utils import feature_to_joints, visualize_motion

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Config
config = Config()
config.dataset_path = Path("./dataset/humanml3d-subset")

Using device: cuda


In [11]:
# Motion-only dataset + dataloader
mean = np.load(config.dataset_path / "Mean.npy")
std = np.load(config.dataset_path / "Std.npy")

train_dataset = Text2MotionDataset(config, mean, std, split="train")
val_dataset = Text2MotionDataset(config, mean, std, split="val")

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=False,
    collate_fn=text2motion_collate_fn,
 )
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
    collate_fn=text2motion_collate_fn,
 )

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

100%|██████████| 500/500 [00:23<00:00, 21.60it/s]


Train batches: 136 | Val batches: 17


In [12]:
# Text encoder (CLIP, frozen)
text_encoder = CLIPEncoder().to(device)
for param in text_encoder.parameters():
    param.requires_grad = False

def encode_text(captions):
    with torch.no_grad():
        emb = text_encoder(captions)
    return emb.to(device)

Loading CLIP model 'openai/clip-vit-base-patch32'...


Loading weights: 100%|██████████| 196/196 [00:00<00:00, 322.09it/s, Materializing param=text_model.final_layer_norm.weight]                    
CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.wei

In [13]:
class TextConditionedMotionGRU(nn.Module):
    """Predict next absolute frame from motion sequence conditioned on text."""

    def __init__(
        self,
        motion_dim: int,
        text_dim: int,
        hidden_dim: int,
        num_layers: int = 1,
        text_scale: float = 1.0,
    ):
        super().__init__()
        self.motion_dim = motion_dim
        self.text_dim = text_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.text_scale = text_scale
        self.text_to_hidden = nn.Linear(text_dim, hidden_dim)
        self.gru = nn.GRU(
            input_size=motion_dim + text_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_dim, motion_dim)

    def forward(self, motion_prev, text_emb):
        """
        motion_prev: (B, T, C) normalized motion inputs
        text_emb: (B, text_dim)
        returns pred_next: (B, T, C)
        """
        bsz, t_len, _ = motion_prev.shape
        text_seq = self.text_scale * text_emb.unsqueeze(1).expand(bsz, t_len, self.text_dim)
        gru_input = torch.cat([motion_prev, text_seq], dim=-1)
        h0 = self.text_to_hidden(text_emb)
        h0 = h0.unsqueeze(0).repeat(self.num_layers, 1, 1)
        h, _ = self.gru(gru_input, h0)
        pred_next = self.head(h)
        return pred_next

In [14]:
# Model + training loop
model = TextConditionedMotionGRU(
    motion_dim=config.motion_dim,
    text_dim=512,
    hidden_dim=512,
    num_layers=2,
    text_scale=0.3,
 ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
bone_loss_lambda = 0.1
p_start = 0.1
p_end = 0.3

mean_t = torch.from_numpy(mean).float().to(device)
std_t = torch.from_numpy(std).float().to(device)

bone_pairs = []
for chain in t2m_kinematic_chain:
    for i in range(len(chain) - 1):
        bone_pairs.append((chain[i], chain[i + 1]))

def bone_length_loss(pred_feat_denorm, target_feat_denorm, mask):
    pred_joints = feature_to_joints(pred_feat_denorm, dataset_type="t2m")
    target_joints = feature_to_joints(target_feat_denorm, dataset_type="t2m")
    if not torch.is_tensor(pred_joints):
        pred_joints = torch.from_numpy(pred_joints).to(pred_feat_denorm.device)
    if not torch.is_tensor(target_joints):
        target_joints = torch.from_numpy(target_joints).to(target_feat_denorm.device)

    bone_diffs = []
    for a_idx, b_idx in bone_pairs:
        pred_len = (pred_joints[:, :, a_idx, :] - pred_joints[:, :, b_idx, :]).norm(dim=-1)
        tgt_len = (target_joints[:, :, a_idx, :] - target_joints[:, :, b_idx, :]).norm(dim=-1)
        bone_diffs.append((pred_len - tgt_len).pow(2))
    bone_diffs = torch.stack(bone_diffs, dim=-1)  # (B, T, B)
    bone_loss = (bone_diffs * mask.unsqueeze(-1)).sum() / mask.sum().clamp_min(1)
    return bone_loss

def compute_loss(batch, sampling_prob=0.0):
    motion = batch["motion"].to(device)  # (B, T, 263) normalized features
    lengths = batch["lengths"].to(device)
    captions = batch["captions"]
    text_emb = encode_text(captions)

    target = motion[:, 1:]
    t_len = target.shape[1]
    bsz = motion.shape[0]

    h = model.text_to_hidden(text_emb).unsqueeze(0).repeat(model.num_layers, 1, 1)
    x_t = motion[:, 0]
    preds = []
    for t in range(t_len):
        text_step = model.text_scale * text_emb.unsqueeze(1)
        gru_input = torch.cat([x_t.unsqueeze(1), text_step], dim=-1)
        h_seq, h = model.gru(gru_input, h)
        pred_next = model.head(h_seq[:, 0, :])
        preds.append(pred_next)

        gt_next = motion[:, t + 1]
        if sampling_prob > 0.0:
            use_pred = (torch.rand(bsz, device=motion.device) < sampling_prob).unsqueeze(1)
            x_t = torch.where(use_pred, pred_next, gt_next)
        else:
            x_t = gt_next

    pred = torch.stack(preds, dim=1)
    mask = torch.arange(t_len, device=motion.device).unsqueeze(0)
    mask = mask < (lengths - 1).unsqueeze(1)
    mse = (pred - target).pow(2).mean(dim=-1)
    loss_mse = (mse * mask).sum() / mask.sum().clamp_min(1)

    pred_denorm = pred * std_t + mean_t
    target_denorm = target * std_t + mean_t
    loss_bone = bone_length_loss(pred_denorm, target_denorm, mask)
    loss = loss_mse + bone_loss_lambda * loss_bone
    return loss

def train_epoch(loader, sampling_prob):
    model.train()
    total_loss = 0.0
    num_batches = 0
    for batch in loader:
        optimizer.zero_grad()
        loss = compute_loss(batch, sampling_prob=sampling_prob)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / max(1, num_batches)

def validate(loader):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    with torch.no_grad():
        for batch in loader:
            loss = compute_loss(batch, sampling_prob=0.0)
            total_loss += loss.item()
            num_batches += 1
    return total_loss / max(1, num_batches)

# Train
num_epochs = 10
for epoch in range(num_epochs):
    if num_epochs > 1:
        sampling_prob = p_start + (p_end - p_start) * (epoch / (num_epochs - 1))
    else:
        sampling_prob = p_start
    train_loss = train_epoch(train_loader, sampling_prob)
    val_loss = validate(val_loader)
    print(
        f"Epoch {epoch+1}/{num_epochs} | p={sampling_prob:.2f} | "
        f"Train: {train_loss:.6f} | Val: {val_loss:.6f}"
    )

Epoch 1/10 | p=0.10 | Train: 0.582482 | Val: 0.343955
Epoch 2/10 | p=0.12 | Train: 0.283340 | Val: 0.222504
Epoch 3/10 | p=0.14 | Train: 0.200433 | Val: 0.170623
Epoch 4/10 | p=0.17 | Train: 0.159742 | Val: 0.141017
Epoch 5/10 | p=0.19 | Train: 0.135592 | Val: 0.122744
Epoch 6/10 | p=0.21 | Train: 0.119956 | Val: 0.108909
Epoch 7/10 | p=0.23 | Train: 0.109086 | Val: 0.099655
Epoch 8/10 | p=0.26 | Train: 0.101030 | Val: 0.091895
Epoch 9/10 | p=0.28 | Train: 0.094758 | Val: 0.086281
Epoch 10/10 | p=0.30 | Train: 0.090274 | Val: 0.081210


In [15]:
def autoregressive_inference(seed_motion, text_emb, steps=None):
    """Generate motion by predicting next absolute frames with text conditioning."""
    model.eval()
    device = next(model.parameters()).device
    seed_motion = torch.as_tensor(seed_motion).float().to(device)
    text_emb = torch.as_tensor(text_emb).float().to(device)
    if seed_motion.dim() != 2:
        raise ValueError("seed_motion must have shape (T, C)")
    if text_emb.dim() != 2:
        raise ValueError("text_emb must have shape (B, text_dim)")
    if steps is None:
        steps = seed_motion.shape[0] - 1

    x_t = seed_motion[0].clone().unsqueeze(0)  # (1, C)
    h = model.text_to_hidden(text_emb).unsqueeze(0).repeat(model.num_layers, 1, 1)
    frames = [x_t.squeeze(0).clone()]

    with torch.no_grad():
        for _ in range(steps):
            text_step = model.text_scale * text_emb.unsqueeze(1)
            gru_input = torch.cat([x_t.unsqueeze(1), text_step], dim=-1)
            h_seq, h = model.gru(gru_input, h)
            x_t = model.head(h_seq[:, 0, :])
            frames.append(x_t.squeeze(0).clone())

    rollout = torch.stack(frames, dim=0)
    return rollout

In [35]:
# Visualize different captions for the same seed motion
sample = val_dataset[0]
seed_motion = sample[2]  # motion (T, C) normalized
gt_joints = sample[3].numpy()

captions = [
    "a person is walking",
    "a person is is turning behind",
    "a person is dancing",
 ]

rollout_steps = min(60, seed_motion.shape[0] - 1)
mean_t = torch.from_numpy(mean).float().to(device)
std_t = torch.from_numpy(std).float().to(device)

for caption in captions:
    text_emb = encode_text([caption])
    generated_motion = autoregressive_inference(seed_motion, text_emb, steps=rollout_steps)
    generated_motion = generated_motion * std_t + mean_t
    generated_joints = feature_to_joints(generated_motion, dataset_type="t2m").cpu().numpy()

    generated_joints_vis = generated_joints * 10.0
    gt_joints_vis = gt_joints[: generated_joints.shape[0]] * 10.0
    root_offset = gt_joints[0, 0, :]
    generated_joints_vis[:, 0, :] += root_offset

    print(f"Caption: {caption}")
    ani = visualize_motion(
        generated_joints_vis,
        ground_truth=gt_joints_vis,
        title=f"Stage 2 Rollout vs GT: {caption}",
        notebook=True,
        fps=20,
        skip_frames=2,
     )
    display(ani)

Caption: a person is walking


Caption: a person is is turning behind


Caption: a person is dancing


In [36]:
# Rollout stats after 200 frames
sample = val_dataset[0]
seed_motion = sample[2]
caption = "a person is walking forward"
text_emb = encode_text([caption])

rollout_steps = min(200, seed_motion.shape[0] - 1)
generated_motion = autoregressive_inference(seed_motion, text_emb, steps=rollout_steps)
generated_motion = generated_motion * torch.from_numpy(std).float().to(device)
generated_motion = generated_motion + torch.from_numpy(mean).float().to(device)

# Motion stats (feature space)
v = generated_motion[1:] - generated_motion[:-1]
a = v[1:] - v[:-1]
v_norm = v.norm(dim=-1)
a_norm = a.norm(dim=-1)
print("Motion Stats")
print(f"  mean velocity norm: {v_norm.mean().item():.6f}")
print(f"  std velocity norm : {v_norm.std(unbiased=False).item():.6f}")
print(f"  mean accel norm   : {a_norm.mean().item():.6f}")
print(f"  std accel norm    : {a_norm.std(unbiased=False).item():.6f}")

# Bone stats (joint space)
joints = feature_to_joints(generated_motion, dataset_type="t2m").cpu().numpy()

bone_pairs = []
for chain in t2m_kinematic_chain:
    for i in range(len(chain) - 1):
        bone_pairs.append((chain[i], chain[i + 1]))

bone_lengths = []
for a_idx, b_idx in bone_pairs:
    diff = joints[:, a_idx, :] - joints[:, b_idx, :]
    bone_lengths.append(np.linalg.norm(diff, axis=-1))
bone_lengths = np.stack(bone_lengths, axis=1)  # (T, B)

bone_std = bone_lengths.std(axis=0)
bone_mean_std = bone_std.mean()
bone_max_std = bone_std.max()
bone_max_dev = np.max(np.abs(bone_lengths - bone_lengths[0]))

print("Bone Stats")
print(f"  mean bone std: {bone_mean_std:.6f}")
print(f"  max bone std : {bone_max_std:.6f}")
print(f"  max deviation: {bone_max_dev:.6f}")

Motion Stats
  mean velocity norm: 0.132021
  std velocity norm : 0.121111
  mean accel norm   : 0.033060
  std accel norm    : 0.047001
Bone Stats
  mean bone std: 0.014670
  max bone std : 0.047361
  max deviation: 0.161979


In [37]:
# Long rollout stability + multi-caption diversity check
sample = val_dataset[0]
seed_motion = sample[2]
rollout_steps = 200  # or min(200, seed_motion.shape[0] - 1)
text_emb = encode_text(["a person is walking forward"])
long_rollout = autoregressive_inference(seed_motion, text_emb, steps=rollout_steps)

# Check bone deviation over the rollout
long_rollout_denorm = long_rollout * std_t + mean_t
joints_long = feature_to_joints(long_rollout_denorm, dataset_type="t2m")
bone_lengths = []
for a_idx, b_idx in bone_pairs:
    bone_len = (joints_long[:, a_idx, :] - joints_long[:, b_idx, :]).norm(dim=-1)
    bone_lengths.append(bone_len)
bone_lengths = torch.stack(bone_lengths, dim=-1)
max_bone_dev = (bone_lengths.max() - bone_lengths.min()).item()
print(f"Max bone deviation over long rollout: {max_bone_dev:.6f}")

# Test multiple captions for diversity
captions = [
    "a person is walking forward",
    "a person is running",
    "a person is jumping",
    "a person is dancing",
    "a person is waving both arms",
    "a person is turning around",
]
rollouts = []
for caption in captions:
    text_emb = encode_text([caption])
    rollout = autoregressive_inference(seed_motion, text_emb, steps=rollout_steps)
    rollouts.append(rollout)
    print(f"Caption: {caption}")

baseline = rollouts[0]
for idx, caption in enumerate(captions[1:], start=1):
    mean_abs_diff = (baseline - rollouts[idx]).abs().mean().item()
    print(f"Mean |diff| vs baseline ({caption}): {mean_abs_diff:.6f}")

Max bone deviation over long rollout: 0.426907
Caption: a person is walking forward
Caption: a person is running
Caption: a person is jumping
Caption: a person is dancing
Caption: a person is waving both arms
Caption: a person is turning around
Mean |diff| vs baseline (a person is running): 0.980183
Mean |diff| vs baseline (a person is jumping): 0.992238
Mean |diff| vs baseline (a person is dancing): 0.701161
Mean |diff| vs baseline (a person is waving both arms): 0.919921
Mean |diff| vs baseline (a person is turning around): 0.793709


In [38]:
# HARD Conditioning Test: same seed, different text
sample = val_dataset[0]
seed_motion = sample[2]  # (T, C) normalized
rollout_steps = min(200, seed_motion.shape[0] - 1)

caption_a = "walk forward"
caption_b = "wave left hand"

text_emb_a = encode_text([caption_a])
text_emb_b = encode_text([caption_b])

generated_a = autoregressive_inference(seed_motion, text_emb_a, steps=rollout_steps)
generated_b = autoregressive_inference(seed_motion, text_emb_b, steps=rollout_steps)

mean_abs_diff = (generated_a - generated_b).abs().mean().item()
print("HARD Conditioning Test")
print(f"  caption A: {caption_a}")
print(f"  caption B: {caption_b}")
print(f"  mean |diff|: {mean_abs_diff:.6f}")

HARD Conditioning Test
  caption A: walk forward
  caption B: wave left hand
  mean |diff|: 0.930900
